In [1]:
print("Here begins the Bayesian networks notebook.")

Here begins the Bayesian networks notebook.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/life_satisfaction_canadian_survey_bn_clean.csv")

df_bn.head()

,Health_region_ grouped,Gender,Marital_status,Household,Age,Worked_job_business,Edu_level,Gen_health_state,Life_satisfaction,Mental_health_state,Stress_level,Work_stress,Sense_belonging,Weight_state,BMI_12_17,BMI_18_above,Sleep_apnea,High_BP,High_cholestrol,Diabetic,Fatigue_syndrome,Mood_disorder,Anxiety_disorder,Respiratory_chronic_con,Musculoskeletal_con,Cardiovascular_con,Health_utility_indx,Pain_status,Act_improve_health,Fruit_veg_con,Smoked,Tobaco_use,weekly_alcohol,Cannabies_use,Drug_use,Total_active_time,Total_physical_act_time,Other_physical_act_time,Physical_vigorous_act_time,Work_hours,working_status,Aboriginal_identity,Birth_country,Immigrant,Insurance_cover,Food_security,Income_source,Total_income
0,47906,2,1.0,2.0,3,1.0,3.0,3.0,9.0,3.0,2.0,2.0,2.0,3.0,NaN,2.0,2.0,2.0,2,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,NaN,bin_0,NaN,NaN,NaN,2.0,2.0,0.0,0.0,60.0,10.0,38.0,1.0,2.0,1.0,2.0,1.0,0.0,1.0,5.0
1,47906,1,1.0,2.0,5,NaN,2.0,3.0,4.0,3.0,3.0,NaN,3.0,1.0,NaN,2.0,1.0,1.0,2,2.0,2.0,1.0,2.0,2.0,2.0,2.0,NaN,1.0,NaN,bin_0,NaN,NaN,NaN,2.0,2.0,0.0,0.0,0.0,0.0,NaN,NaN,2.0,1.0,2.0,1.0,0.0,2.0,4.0
2,59914,2,2.0,1.0,5,NaN,1.0,2.0,7.0,3.0,3.0,NaN,2.0,1.0,NaN,2.0,2.0,1.0,2,1.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,NaN,bin_1,NaN,2.0,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,2.0,6.0,NaN,2.0,2.0
3,13904,1,2.0,1.0,5,NaN,1.0,3.0,8.0,3.0,3.0,NaN,2.0,1.0,NaN,2.0,2.0,1.0,2,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,NaN,bin_1,NaN,NaN,NaN,2.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,2.0,6.0,0.0,2.0,3.0
4,46903,1,2.0,1.0,4,2.0,3.0,5.0,0.0,5.0,4.0,NaN,3.0,3.0,NaN,2.0,2.0,1.0,2,NaN,2.0,2.0,2.0,NaN,1.0,NaN,1.0,2.0,NaN,bin_1,NaN,NaN,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,2.0,2.0,0.0,2.0,1.0


In [2]:
# Inspect shape, data types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(108252, 48)
Health_region_ grouped          int64
Gender                          int64
Marital_status                float64
Household                     float64
Age                             int64
Worked_job_business           float64
Edu_level                     float64
Gen_health_state              float64
Life_satisfaction             float64
Mental_health_state           float64
Stress_level                  float64
Work_stress                   float64
Sense_belonging               float64
Weight_state                  float64
BMI_12_17                     float64
BMI_18_above                  float64
Sleep_apnea                   float64
High_BP                       float64
High_cholestrol                 int64
Diabetic                      float64
Fatigue_syndrome              float64
Mood_disorder                 float64
Anxiety_disorder              float64
Respiratory_chronic_con       float64
Musculoskeletal_con           float64
Cardiovascular_con            float64

In [3]:
# Convert all columns to string (discrete states)
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Health_region_ grouped        string
Gender                        string
Marital_status                string
Household                     string
Age                           string
Worked_job_business           string
Edu_level                     string
Gen_health_state              string
Life_satisfaction             string
Mental_health_state           string
Stress_level                  string
Work_stress                   string
Sense_belonging               string
Weight_state                  string
BMI_12_17                     string
BMI_18_above                  string
Sleep_apnea                   string
High_BP                       string
High_cholestrol               string
Diabetic                      string
Fatigue_syndrome              string
Mood_disorder                 string
Anxiety_disorder              string
Respiratory_chronic_con       string
Musculoskeletal_con           string
Cardiovascular_con            string
Health_utility_indx           string
P

In [4]:
# Check the number of unique states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
35,Total_active_time,211
36,Total_physical_act_time,208
37,Other_physical_act_time,202
38,Physical_vigorous_act_time,156
0,Health_region_ grouped,91
32,weekly_alcohol,80
30,Smoked,51
39,Work_hours,50
8,Life_satisfaction,11
18,High_cholestrol,5


In [11]:
# Create the BN modeling dataset
df_bn_model = df_bn.copy()

# Drop high-cardinality region identifier
df_bn_model = df_bn_model.drop(columns=["Health_region_ grouped"], errors="ignore")

# Discretize time variables into bins
time_cols = [
    "Total_active_time",
    "Total_physical_act_time",
    "Other_physical_act_time",
    "Physical_vigorous_act_time"
]

bin_name_map = {
    2: ["low", "high"],
    3: ["low", "medium", "high"],
    4: ["low", "medium_low", "medium_high", "high"],
    5: ["very_low", "low", "medium", "high", "very_high"]
}

for col in time_cols:
    if col in df_bn_model.columns:
        s = pd.to_numeric(df_bn_model[col], errors="coerce")

        if s.nunique(dropna=True) > 20:
            # First create numeric bins, allowing qcut to drop duplicate edges
            binned = pd.qcut(
                s,
                q=5,
                labels=False,
                duplicates="drop"
            )

            # Convert to nullable integer so NaNs are preserved
            binned = binned.astype("Int64")

            # Determine how many bins were actually created
            n_bins = binned.dropna().nunique()

            # Get appropriate labels
            labels = bin_name_map.get(
                n_bins,
                [f"bin_{i}" for i in range(n_bins)]
            )

            # Map numeric bins to string labels
            df_bn_model[f"{col}_binned"] = binned.map(
                lambda x: labels[int(x)] if pd.notna(x) else pd.NA
            ).astype("string")

            # Drop original high-cardinality column
            df_bn_model = df_bn_model.drop(columns=[col])

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(108252, 47)
['Gender', 'Marital_status', 'Household', 'Age', 'Worked_job_business', 'Edu_level', 'Gen_health_state', 'Life_satisfaction', 'Mental_health_state', 'Stress_level', 'Work_stress', 'Sense_belonging', 'Weight_state', 'BMI_12_17', 'BMI_18_above', 'Sleep_apnea', 'High_BP', 'High_cholestrol', 'Diabetic', 'Fatigue_syndrome', 'Mood_disorder', 'Anxiety_disorder', 'Respiratory_chronic_con', 'Musculoskeletal_con', 'Cardiovascular_con', 'Health_utility_indx', 'Pain_status', 'Act_improve_health', 'Fruit_veg_con', 'Smoked', 'Tobaco_use', 'weekly_alcohol', 'Cannabies_use', 'Drug_use', 'Work_hours', 'working_status', 'Aboriginal_identity', 'Birth_country', 'Immigrant', 'Insurance_cover', 'Food_security', 'Income_source', 'Total_income', 'Total_active_time_binned', 'Total_physical_act_time_binned', 'Other_physical_act_time_binned', 'Physical_vigorous_act_time_binned']


In [12]:
for col in df_bn_model.columns:
    if col.endswith("_binned"):
        print(f"\n{col}:")
        print(df_bn_model[col].value_counts(dropna=False))


Total_active_time_binned:
Total_active_time_binned
<NA>    87474
low     16873
high     3905
Name: count, dtype: Int64

Total_physical_act_time_binned:
Total_physical_act_time_binned
<NA>    87491
low     16678
high     4083
Name: count, dtype: Int64

Other_physical_act_time_binned:
Other_physical_act_time_binned
<NA>    87520
low     16679
high     4053
Name: count, dtype: Int64

Physical_vigorous_act_time_binned:
Physical_vigorous_act_time_binned
<NA>     87728
bin_0    20524
Name: count, dtype: Int64


In [13]:
df_bn_model = df_bn_model.drop(columns=[
    "Total_active_time_binned",
    "Total_physical_act_time_binned",
    "Other_physical_act_time_binned",
    "Physical_vigorous_act_time_binned"
], errors="ignore")

In [14]:
# Check number of states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn_model.columns,
    "Num_States": [df_bn_model[col].nunique(dropna=True) for col in df_bn_model.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
31,weekly_alcohol,80
29,Smoked,51
34,Work_hours,50
7,Life_satisfaction,11
9,Stress_level,5
6,Gen_health_state,5
42,Total_income,5
17,High_cholestrol,5
10,Work_stress,5
3,Age,5


In [16]:
# Drop weekly_alcohol completely - many invalid values and high cardinality

# Drop weekly_alcohol
df_bn_model = df_bn_model.drop(columns=["weekly_alcohol"], errors="ignore")

# Discretize high-cardinality numeric variables
high_card_cols = ["Smoked", "Work_hours"]

for col in high_card_cols:
    if col in df_bn_model.columns:
        s = pd.to_numeric(df_bn_model[col], errors="coerce")

        if s.nunique(dropna=True) > 20:
            binned = pd.qcut(
                s,
                q=5,
                labels=False,
                duplicates="drop"
            )

            binned = binned.astype("Int64")

            n_bins = binned.dropna().nunique()

            label_pool = ["very_low", "low", "medium", "high", "very_high"]
            labels = label_pool[:n_bins]

            df_bn_model[f"{col}_binned"] = binned.map(
                lambda x: labels[int(x)] if pd.notna(x) else pd.NA
            ).astype("string")

            # Drop original high-cardinality column
            df_bn_model = df_bn_model.drop(columns=[col])

print("BN modeling dataset shape:", df_bn_model.shape)
print("\nColumns:")
print(df_bn_model.columns.tolist())

print("\nDiscretized variables:")
for col in df_bn_model.columns:
    if col.endswith("_binned"):
        print(f"\n{col}:")
        print(df_bn_model[col].value_counts(dropna=False))

BN modeling dataset shape: (108252, 42)

Columns:
['Gender', 'Marital_status', 'Household', 'Age', 'Worked_job_business', 'Edu_level', 'Gen_health_state', 'Life_satisfaction', 'Mental_health_state', 'Stress_level', 'Work_stress', 'Sense_belonging', 'Weight_state', 'BMI_12_17', 'BMI_18_above', 'Sleep_apnea', 'High_BP', 'High_cholestrol', 'Diabetic', 'Fatigue_syndrome', 'Mood_disorder', 'Anxiety_disorder', 'Respiratory_chronic_con', 'Musculoskeletal_con', 'Cardiovascular_con', 'Health_utility_indx', 'Pain_status', 'Act_improve_health', 'Fruit_veg_con', 'Tobaco_use', 'Cannabies_use', 'Drug_use', 'working_status', 'Aboriginal_identity', 'Birth_country', 'Immigrant', 'Insurance_cover', 'Food_security', 'Income_source', 'Total_income', 'Smoked_binned', 'Work_hours_binned']

Discretized variables:

Smoked_binned:
Smoked_binned
<NA>         96158
low           3007
very_low      2873
medium        2457
very_high     1994
high          1763
Name: count, dtype: Int64

Work_hours_binned:
Work_hou

In [17]:
# Check number of states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn_model.columns,
    "Num_States": [df_bn_model[col].nunique(dropna=True) for col in df_bn_model.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
7,Life_satisfaction,11
3,Age,5
9,Stress_level,5
8,Mental_health_state,5
6,Gen_health_state,5
36,Insurance_cover,5
39,Total_income,5
40,Smoked_binned,5
17,High_cholestrol,5
10,Work_stress,5


In [41]:
# Optional final cleanup for BN stability
df_bn_model = df_bn_model.drop(columns=["Smoked_binned", "BMI_12_17", "Act_improve_health", "Tobaco_use"], errors="ignore")

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(108252, 38)
['Gender', 'Marital_status', 'Household', 'Age', 'Worked_job_business', 'Edu_level', 'Gen_health_state', 'Life_satisfaction', 'Mental_health_state', 'Stress_level', 'Work_stress', 'Sense_belonging', 'Weight_state', 'BMI_18_above', 'Sleep_apnea', 'High_BP', 'High_cholestrol', 'Diabetic', 'Fatigue_syndrome', 'Mood_disorder', 'Anxiety_disorder', 'Respiratory_chronic_con', 'Musculoskeletal_con', 'Cardiovascular_con', 'Health_utility_indx', 'Pain_status', 'Fruit_veg_con', 'Cannabies_use', 'Drug_use', 'working_status', 'Aboriginal_identity', 'Birth_country', 'Immigrant', 'Insurance_cover', 'Food_security', 'Income_source', 'Total_income', 'Work_hours_binned']


In [42]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import HillClimbSearch, ExpertKnowledge, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [43]:
# Learn an unconstrained Bayesian Network structure
# - bic-d is for discrete data
# - max_indegree limits complexity
hc = HillClimbSearch(df_bn_model)

learned_dag = hc.estimate(
    scoring_method="bic-d",
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag.nodes()))
print("Number of edges:", len(learned_dag.edges()))
print("\nLearned edges:")
print(sorted(learned_dag.edges()))

Number of nodes: 38
Number of edges: 103

Learned edges:
[('Age', 'Musculoskeletal_con'), ('BMI_18_above', 'Fruit_veg_con'), ('BMI_18_above', 'Weight_state'), ('Birth_country', 'Immigrant'), ('Cardiovascular_con', 'Respiratory_chronic_con'), ('Food_security', 'Aboriginal_identity'), ('Food_security', 'Age'), ('Food_security', 'Anxiety_disorder'), ('Food_security', 'BMI_18_above'), ('Food_security', 'Birth_country'), ('Food_security', 'Cannabies_use'), ('Food_security', 'Drug_use'), ('Food_security', 'Edu_level'), ('Food_security', 'Gen_health_state'), ('Food_security', 'Gender'), ('Food_security', 'Health_utility_indx'), ('Food_security', 'High_BP'), ('Food_security', 'High_cholestrol'), ('Food_security', 'Household'), ('Food_security', 'Life_satisfaction'), ('Food_security', 'Mental_health_state'), ('Food_security', 'Mood_disorder'), ('Food_security', 'Pain_status'), ('Food_security', 'Sense_belonging'), ('Food_security', 'Sleep_apnea'), ('Food_security', 'Stress_level'), ('Food_secur

In [44]:
# Convert the learned graph into a discrete Bayesian Network
bn_unconstrained = DiscreteBayesianNetwork(learned_dag.edges())

# Fit CPDs using Bayesian parameter estimation
# BDeu prior provides smoothing and avoids unstable zero probabilities
bn_unconstrained.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_unconstrained.check_model())

Model check: True


In [45]:
# Show the direct parent variables of Life_satisfaction
if "Life_satisfaction" in bn_unconstrained.nodes():
    print("Parents of Life_satisfaction in unconstrained BN:")
    print(list(bn_unconstrained.predecessors("Life_satisfaction")))

# Inspect CPD metadata for the target variable
if "Life_satisfaction" in bn_unconstrained.nodes():
    cpd_ls = bn_unconstrained.get_cpds("Life_satisfaction")
    print("CPD state names for Life_satisfaction:")
    print(cpd_ls.state_names)

Parents of Life_satisfaction in unconstrained BN:
['Food_security', 'Income_source', 'Work_hours_binned']
CPD state names for Life_satisfaction:
{'Life_satisfaction': ['0.0', '1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0'], 'Food_security': ['0.0', '1.0', '2.0', '3.0'], 'Income_source': ['1.0', '2.0'], 'Work_hours_binned': ['high', 'low', 'medium', 'very_low']}


In [47]:
# Create inference engine for the unconstrained BN
infer_unconstrained = VariableElimination(bn_unconstrained)  # type: ignore

In [48]:
# Check valid state values for important variables before running inference
for col in [
    "Life_satisfaction",
    "Mental_health_state",
    "Stress_level",
    "Sense_belonging",
    "Gen_health_state",
    "Food_security",
    "Total_income",
    "Work_hours_binned"
]:
    if col in df_bn_model.columns:
        print(f"\n{col}:")
        print(sorted(df_bn_model[col].dropna().unique()))


Life_satisfaction:
['0.0', '1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0']

Mental_health_state:
['1.0', '2.0', '3.0', '4.0', '5.0']

Stress_level:
['1.0', '2.0', '3.0', '4.0', '5.0']

Sense_belonging:
['1.0', '2.0', '3.0', '4.0']

Gen_health_state:
['1.0', '2.0', '3.0', '4.0', '5.0']

Food_security:
['0.0', '1.0', '2.0', '3.0']

Total_income:
['1.0', '2.0', '3.0', '4.0', '5.0']

Work_hours_binned:
['high', 'low', 'medium', 'very_low']


In [49]:
# Example query 1:
# Probability distribution of Life_satisfaction under good mental health and low stress
q1 = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "1.0",   # excellent
        "Stress_level": "1.0"           # not at all stressful
    },
    show_progress=False
)

print(q1)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0016 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0009 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.1939 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0021 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0037 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0069 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0276 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.0441 |
+---------

In [50]:
# Example query 2:
# Probability distribution of Life_satisfaction under poor mental health and high stress
q2 = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "5.0",   # poor
        "Stress_level": "5.0"           # extremely stressful
    },
    show_progress=False
)

print(q2)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0052 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0022 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.1422 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0109 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0129 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0248 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0843 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.0871 |
+---------

In [51]:
# Example query 3:
# Compare Life_satisfaction under strong versus weak sense of belonging
q3_strong = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

q3_weak = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "4.0"        # very weak
    },
    show_progress=False
)

print("Life_satisfaction with very strong sense of belonging:")
print(q3_strong)

print("\nLife_satisfaction with very weak sense of belonging:")
print(q3_weak)

Life_satisfaction with very strong sense of belonging:
+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0019 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0010 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.1900 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0028 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0042 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0082 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0320 |
+-------------------------+--------------------------+
| Life_sat

In [52]:
# Summarize key probability mass for easier interpretation
def summarize_life_satisfaction(query_result):
    probs = dict(zip(
        [str(state) for state in query_result.state_names["Life_satisfaction"]],
        query_result.values
    ))

    low = sum(probs.get(str(x), 0) for x in ["0.0", "1.0", "2.0", "3.0", "4.0"])
    medium = sum(probs.get(str(x), 0) for x in ["5.0", "6.0", "7.0"])
    high = sum(probs.get(str(x), 0) for x in ["8.0", "9.0", "10.0"])

    return pd.Series({
        "Low (0-4)": low,
        "Medium (5-7)": medium,
        "High (8-10)": high
    })

print("Query 1 summary:")
print(summarize_life_satisfaction(q1))

print("\nQuery 2 summary:")
print(summarize_life_satisfaction(q2))

print("\nStrong belonging summary:")
print(summarize_life_satisfaction(q3_strong))

print("\nWeak belonging summary:")
print(summarize_life_satisfaction(q3_weak))

Query 1 summary:
Low (0-4)       0.015257
Medium (5-7)    0.230253
High (8-10)     0.754490
dtype: float64

Query 2 summary:
Low (0-4)       0.056071
Medium (5-7)    0.369524
High (8-10)     0.574405
dtype: float64

Strong belonging summary:
Low (0-4)       0.018024
Medium (5-7)    0.242523
High (8-10)     0.739453
dtype: float64

Weak belonging summary:
Low (0-4)       0.023429
Medium (5-7)    0.264286
High (8-10)     0.712285
dtype: float64


In [53]:
# Build a restricted BN with expert constraints
# The goal is to avoid clearly implausible directions and make Life_satisfaction
# more interpretable as an outcome rather than a cause of demographics or structure.

forbidden_edges = [
    # Life satisfaction should not cause demographics
    ("Life_satisfaction", "Gender"),
    ("Life_satisfaction", "Age"),
    ("Life_satisfaction", "Marital_status"),
    ("Life_satisfaction", "Household"),
    ("Life_satisfaction", "Birth_country"),
    ("Life_satisfaction", "Immigrant"),
    ("Life_satisfaction", "Aboriginal_identity"),

    # Life satisfaction should not cause socioeconomic variables
    ("Life_satisfaction", "Edu_level"),
    ("Life_satisfaction", "Income_source"),
    ("Life_satisfaction", "Total_income"),
    ("Life_satisfaction", "Food_security"),
    ("Life_satisfaction", "Insurance_cover"),
    ("Life_satisfaction", "working_status"),
    ("Life_satisfaction", "Work_hours_binned"),
    ("Life_satisfaction", "Worked_job_business"),

    # Life satisfaction should not cause chronic conditions / health status
    ("Life_satisfaction", "Gen_health_state"),
    ("Life_satisfaction", "Mental_health_state"),
    ("Life_satisfaction", "Stress_level"),
    ("Life_satisfaction", "Work_stress"),
    ("Life_satisfaction", "Sense_belonging"),
    ("Life_satisfaction", "BMI_18_above"),
    ("Life_satisfaction", "Weight_state"),
    ("Life_satisfaction", "Sleep_apnea"),
    ("Life_satisfaction", "High_BP"),
    ("Life_satisfaction", "High_cholestrol"),
    ("Life_satisfaction", "Diabetic"),
    ("Life_satisfaction", "Fatigue_syndrome"),
    ("Life_satisfaction", "Mood_disorder"),
    ("Life_satisfaction", "Anxiety_disorder"),
    ("Life_satisfaction", "Respiratory_chronic_con"),
    ("Life_satisfaction", "Musculoskeletal_con"),
    ("Life_satisfaction", "Cardiovascular_con"),
    ("Life_satisfaction", "Health_utility_indx"),
    ("Life_satisfaction", "Pain_status"),

    # Downstream behaviors should not determine demographics
    ("Mental_health_state", "Gender"),
    ("Mental_health_state", "Age"),
    ("Stress_level", "Gender"),
    ("Stress_level", "Age"),
    ("Sense_belonging", "Gender"),
    ("Sense_belonging", "Age"),
    ("Gen_health_state", "Gender"),
    ("Gen_health_state", "Age"),

    # Economic/contextual variables should not be caused by health outcomes
    ("Mental_health_state", "Income_source"),
    ("Mental_health_state", "Total_income"),
    ("Stress_level", "Income_source"),
    ("Stress_level", "Total_income"),
    ("Gen_health_state", "Income_source"),
    ("Gen_health_state", "Total_income"),
]

expert_knowledge = ExpertKnowledge(
    forbidden_edges=forbidden_edges
)

hc_restricted = HillClimbSearch(df_bn_model)

learned_dag_restricted = hc_restricted.estimate(
    scoring_method="bic-d",
    expert_knowledge=expert_knowledge,
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag_restricted.nodes()))
print("Number of edges:", len(learned_dag_restricted.edges()))
print("\nRestricted BN edges:")
print(sorted(learned_dag_restricted.edges()))

Number of nodes: 38
Number of edges: 103

Restricted BN edges:
[('Age', 'Musculoskeletal_con'), ('BMI_18_above', 'Fruit_veg_con'), ('BMI_18_above', 'Weight_state'), ('Birth_country', 'Immigrant'), ('Cardiovascular_con', 'Respiratory_chronic_con'), ('Food_security', 'Aboriginal_identity'), ('Food_security', 'Age'), ('Food_security', 'Anxiety_disorder'), ('Food_security', 'BMI_18_above'), ('Food_security', 'Birth_country'), ('Food_security', 'Cannabies_use'), ('Food_security', 'Drug_use'), ('Food_security', 'Edu_level'), ('Food_security', 'Gen_health_state'), ('Food_security', 'Gender'), ('Food_security', 'Health_utility_indx'), ('Food_security', 'High_BP'), ('Food_security', 'High_cholestrol'), ('Food_security', 'Household'), ('Food_security', 'Life_satisfaction'), ('Food_security', 'Mental_health_state'), ('Food_security', 'Mood_disorder'), ('Food_security', 'Pain_status'), ('Food_security', 'Sense_belonging'), ('Food_security', 'Sleep_apnea'), ('Food_security', 'Stress_level'), ('Food

In [54]:
# Fit the restricted BN
bn_restricted = DiscreteBayesianNetwork(learned_dag_restricted.edges())

bn_restricted.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Restricted model check:", bn_restricted.check_model())

Restricted model check: True


In [55]:
# Inspect direct parents of Life_satisfaction in the restricted BN
if "Life_satisfaction" in bn_restricted.nodes():
    print("Parents of Life_satisfaction in restricted BN:")
    print(list(bn_restricted.predecessors("Life_satisfaction")))

# Inspect CPD metadata for the target variable in the restricted BN
if "Life_satisfaction" in bn_restricted.nodes():
    cpd_ls_restricted = bn_restricted.get_cpds("Life_satisfaction")
    print("Restricted CPD state names for Life_satisfaction:")
    print(cpd_ls_restricted.state_names)

Parents of Life_satisfaction in restricted BN:
['Food_security', 'Income_source', 'Work_hours_binned']
Restricted CPD state names for Life_satisfaction:
{'Life_satisfaction': ['0.0', '1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0'], 'Food_security': ['0.0', '1.0', '2.0', '3.0'], 'Income_source': ['1.0', '2.0'], 'Work_hours_binned': ['high', 'low', 'medium', 'very_low']}


In [56]:
# Create inference engine for the restricted BN
infer_restricted = VariableElimination(bn_restricted)  # type: ignore

In [57]:
# Query 4:
# Probability distribution of Life_satisfaction under favorable psychosocial conditions
q4 = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "1.0",   # excellent
        "Gen_health_state": "1.0",      # excellent
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

print(q4)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0016 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0009 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.1962 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0021 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0036 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0068 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0271 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.0433 |
+---------

In [58]:
# Query 5:
# Probability distribution of Life_satisfaction under adverse psychosocial conditions
q5 = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "5.0",   # poor
        "Gen_health_state": "5.0",      # poor
        "Food_security": "3.0"          # severely food insecure
    },
    show_progress=False
)

print(q5)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0135 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0035 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.0913 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0240 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0382 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0530 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.1770 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.1354 |
+---------

In [59]:
# Query 6:
# Compare life satisfaction under strong versus weak sense of belonging in the restricted BN
q6_strong = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

q6_weak = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "4.0"        # very weak
    },
    show_progress=False
)

print("Restricted BN - Life_satisfaction with very strong sense of belonging:")
print(q6_strong)

print("\nRestricted BN - Life_satisfaction with very weak sense of belonging:")
print(q6_weak)

Restricted BN - Life_satisfaction with very strong sense of belonging:
+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0019 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0010 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.1900 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0028 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0042 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0082 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0320 |
+-------------------------+----------------------

In [60]:
# Optional helper:
# Summarize the 0-10 Life_satisfaction distribution into low / medium / high groups
def summarize_life_satisfaction(query_result):
    probs = dict(zip(
        [str(state) for state in query_result.state_names["Life_satisfaction"]],
        query_result.values
    ))

    low = sum(probs.get(x, 0) for x in ["0.0", "1.0", "2.0", "3.0", "4.0"])
    medium = sum(probs.get(x, 0) for x in ["5.0", "6.0", "7.0"])
    high = sum(probs.get(x, 0) for x in ["8.0", "9.0", "10.0"])

    return pd.Series({
        "Low (0-4)": low,
        "Medium (5-7)": medium,
        "High (8-10)": high
    })

print("Unconstrained BN - Query 1 summary:")
print(summarize_life_satisfaction(q1))

print("\nUnconstrained BN - Query 2 summary:")
print(summarize_life_satisfaction(q2))

print("\nUnconstrained BN - Strong belonging summary:")
print(summarize_life_satisfaction(q3_strong))

print("\nUnconstrained BN - Weak belonging summary:")
print(summarize_life_satisfaction(q3_weak))

print("\nRestricted BN - Query 4 summary:")
print(summarize_life_satisfaction(q4))

print("\nRestricted BN - Query 5 summary:")
print(summarize_life_satisfaction(q5))

print("\nRestricted BN - Strong belonging summary:")
print(summarize_life_satisfaction(q6_strong))

print("\nRestricted BN - Weak belonging summary:")
print(summarize_life_satisfaction(q6_weak))

Unconstrained BN - Query 1 summary:
Low (0-4)       0.015257
Medium (5-7)    0.230253
High (8-10)     0.754490
dtype: float64

Unconstrained BN - Query 2 summary:
Low (0-4)       0.056071
Medium (5-7)    0.369524
High (8-10)     0.574405
dtype: float64

Unconstrained BN - Strong belonging summary:
Low (0-4)       0.018024
Medium (5-7)    0.242523
High (8-10)     0.739453
dtype: float64

Unconstrained BN - Weak belonging summary:
Low (0-4)       0.023429
Medium (5-7)    0.264286
High (8-10)     0.712285
dtype: float64

Restricted BN - Query 4 summary:
Low (0-4)       0.014906
Medium (5-7)    0.227672
High (8-10)     0.757421
dtype: float64

Restricted BN - Query 5 summary:
Low (0-4)       0.132252
Medium (5-7)    0.519427
High (8-10)     0.348321
dtype: float64

Restricted BN - Strong belonging summary:
Low (0-4)       0.018024
Medium (5-7)    0.242523
High (8-10)     0.739453
dtype: float64

Restricted BN - Weak belonging summary:
Low (0-4)       0.023429
Medium (5-7)    0.264286
High 